# Movie Review Classifier - SVM
This notebook builds a 3-class text classifier using a Support Vector Machine (SVM). We compare the results against theLogistic Regression baseline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

## 1. Data Loading and Preprocessing

Same preprocessing as the Logistic Regression notebook -- drop missing TEXT from training, fill missing TEXT in test with empty strings.

In [2]:
class TextPreprocessor:
    """handles loading and cleaning the data"""
    
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path
    
    def load(self):
        train = pd.read_csv(self.train_path)
        test = pd.read_csv(self.test_path)
        
        train = train.dropna(subset=["TEXT"])  # drop rows with missing text
        test = test.fillna({"TEXT": ""})        # fill missing test text with empty string
        
        return train, test


# load data
preprocessor = TextPreprocessor("../train.csv", "../test.csv")
train, test = preprocessor.load()

X_train = train["TEXT"]
y_train = train["LABEL"]
X_test = test["TEXT"]

print("Train size:", len(train))
print("Test size:", len(test))
print("Class distribution:\n", y_train.value_counts())

Train size: 70310
Test size: 17580
Class distribution:
 LABEL
0    32282
1    19139
2    18889
Name: count, dtype: int64


## 2. Feature Extraction

Same TF-IDF setup as before -- unigrams and bigrams with a max of 50,000 features.

In [3]:
class FeatureExtractor:
    """converts raw text into TF-IDF features"""
    
    def __init__(self, ngram_range=(1, 2), max_features=50000):
        self.vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    
    def fit(self, texts):
        self.vectorizer.fit(texts)
    
    def transform(self, texts):
        return self.vectorizer.transform(texts)
    
    def fit_transform(self, texts):
        return self.vectorizer.fit_transform(texts)

## 3. Classifier

Using a Linear SVM (`LinearSVC`) which is well suited for high-dimensional text classification tasks. Key settings:
- `class_weight="balanced"` to account for class imbalance
- `max_iter=2000` to ensure convergence
- `random_state=66` for reproducibility

In [4]:
class MovieReviewClassifier:
    """wraps LinearSVC for 3-class classification"""
    
    def __init__(self):
        self.clf = LinearSVC(
            class_weight="balanced",  # accounts for class imbalance
            max_iter=2000,
            random_state=66
        )
    
    def train(self, features, labels):
        self.clf.fit(features, labels)
    
    def predict(self, features):
        return self.clf.predict(features)
    
    def evaluate(self, X, y, cv=5):
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
            ("clf", LinearSVC(class_weight="balanced", max_iter=2000, random_state=42))
        ])
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
        scores = cross_val_score(pipeline, X, y, cv=skf, scoring="f1_macro")
        return scores

## 4. Cross Validation

Evaluating using 5-fold stratified cross validation and report the macro F1 score.

In [5]:
classifier = MovieReviewClassifier()
scores = classifier.evaluate(X_train, y_train)

print("SVM CV Macro F1 scores:", scores)
print("Mean:", scores.mean().round(4))
print("Std:", scores.std().round(4))

SVM CV Macro F1 scores: [0.91709665 0.92120638 0.92205261 0.92029626 0.91951198]
Mean: 0.92
Std: 0.0017


### Cross Validation Results

I evaluated the SVM using 5-fold stratified cross validation and got a mean macro F1 of **0.9200** with a standard deviation of **0.0017**. The low standard deviation tells me the model is pretty consistent across folds, which is a good sign. Compared to my Logistic Regression model (mean F1 = 0.9197), the SVM performs about the same -- essentially tied. This makes sense since both models are linear classifiers working on the same TF-IDF features.

I used `LinearSVC` instead of the standard `SVC` because it's much faster on high-dimensional sparse data like TF-IDF matrices, and for text classification tasks they tend to perform similarly. You can read more about `LinearSVC` in the sklearn documentation here: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html and about SVMs for text classification in this sklearn user guide: https://scikit-learn.org/stable/modules/svm.html#text-feature-extraction-and-tfidf-weighting

## 5. Training and Prediction

In [6]:
extractor = FeatureExtractor()
X_train_features = extractor.fit_transform(X_train)
X_test_features = extractor.transform(X_test)

classifier.train(X_train_features, y_train)
predictions = classifier.predict(X_test_features)

print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

Prediction distribution:
0    8046
1    4875
2    4659
Name: count, dtype: int64


### Prediction Distribution

The model predicted 8,046 instances as class 0 (not a review), 4,875 as class 1 (positive), and 4,659 as class 2 (negative). This roughly mirrors the class distribution I observed in the training data during EDA, where class 0 was the largest class at around 46%. The fact that the predictions are proportionally distributed is a good sign that `class_weight="balanced"` is doing its job -- the model isn't just collapsing everything into the majority class.

I used `class_weight="balanced"` based on the documentation: https://scikit-learn.org/stable/modules/svm.html#unbalanced-problems, which recommends this setting when class sizes differ significantly.

## 6. Generate Submission File

In [7]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "LABEL": predictions
})

submission.to_csv("../submission_svm.csv", index=False)
print("Submission saved to submission_svm.csv!")
print(submission.head())

Submission saved to submission_svm.csv!
                     ID  LABEL
0   1087873697464833975      1
1   5853461517618045821      1
2   1225516091890726770      2
3  11795874926011119457      0
4  15956464546963841646      0
